# BIST local LLM: train, evaluate, use (Google Colab)

Runtime: **T4 GPU** (Runtime -> Change runtime type). Run the cells top to bottom. This notebook is self-contained: it trains the QLoRA adapter, evaluates it against the base model, and exports the result. Data preparation is in `notebooks/05_local_llm_finetuning.ipynb` (see `notebooks/README.md`).

## 0. Prepare data (once, on your own machine, not in Colab)
```bash
# 1. generate candidate summaries from the deterministic harness (downloads public OHLCV data)
python -m src.llm.build_corpus --download --dates-per-ticker 3 --output-dir data/llm/candidates_v2 --reviewers Name1 Name2 Name3
# 2. the group reviews data/llm/candidates_v2/candidates_review.csv (human_reviewed=true, reviewer, reviewed_summary)
# 3. convert approved rows into the training file
python -m src.llm.prepare_reviewed_records --candidates data/llm/candidates_v2/candidates.jsonl --review-csv data/llm/candidates_v2/candidates_review.csv
```
Outputs: `candidates.jsonl` (unreviewed drafts, for the trial) and `data/llm/reviewed_records.jsonl` (reviewed, for the real run). Data is not in Git, so upload it in the upload cell below.

## Modes
* `BIST_LLM_TRIAL=1` **trial**: unreviewed drafts + seed records. Pipeline test only, not valid for submission.
* `BIST_LLM_TRIAL=0` **real run**: only `reviewed_records.jsonl`.
* `BIST_LLM_MODEL`: `Qwen/Qwen3-1.7B` (about 20 min on a T4) or `Qwen/Qwen3-4B` (slower).

In [ ]:
!git clone https://github.com/zeyneptanrivermis/bist-harness.git
%cd /content/bist-harness

!git fetch origin
!git switch defne
!git pull origin defne

In [ ]:
!pip install -q -r requirements-llm.txt

In [ ]:
import os
os.environ["BIST_LLM_MODEL"] = "Qwen/Qwen3-1.7B"   # or "Qwen/Qwen3-4B"
os.environ["BIST_LLM_TRIAL"] = "1"                  # "1" = trial (unreviewed), "0" = real (reviewed_records.jsonl)
ADAPTER = ("artifacts/llm/qwen_bist_lora_TRIAL_unreviewed/adapter" if os.environ["BIST_LLM_TRIAL"] == "1"
           else "artifacts/llm/qwen_bist_lora/adapter")
os.environ["BIST_LLM_ADAPTER"] = ADAPTER
print(os.environ["BIST_LLM_MODEL"], os.environ["BIST_LLM_TRIAL"], ADAPTER)

In [ ]:
import os, shutil
from google.colab import files
# Trial: upload candidates.jsonl.  Real run: upload reviewed_records.jsonl.
os.makedirs("data/llm/candidates_v2", exist_ok=True)
for name in files.upload():
    dest = "data/llm/reviewed_records.jsonl" if name.startswith("reviewed_records") else "data/llm/candidates_v2/candidates.jsonl"
    shutil.move(name, dest); print("saved", dest)

## Train

In [ ]:
import json, random, os
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from src.llm.training_data import generate_seed_records, stable_split, validate_records

SEED = 481
random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select a T4 GPU runtime (Runtime -> Change runtime type)."
MODEL_NAME = os.environ["BIST_LLM_MODEL"]
TRIAL = os.environ["BIST_LLM_TRIAL"] == "1"
OUTPUT_DIR = Path(ADAPTER).parent

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

if TRIAL:  # pipeline test on UNREVIEWED deterministic drafts + A-E seeds; not valid for submission
    reviewed_records = read_jsonl("data/llm/candidates_v2/candidates.jsonl") + generate_seed_records()
    for record in reviewed_records:
        record["metadata"]["trial_unreviewed"] = True
    validate_records(reviewed_records, require_human_review=False)
    print(f"TRIAL MODE: {len(reviewed_records)} unreviewed records -> {OUTPUT_DIR}")
else:
    reviewed_records = read_jsonl("data/llm/reviewed_records.jsonl")
    validate_records(reviewed_records, require_human_review=True)
    print(f"Using {len(reviewed_records)} reviewed records -> {OUTPUT_DIR}")

In [ ]:
# QLoRA training (about 20 min for 1.7B on a T4)
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from src.llm.training_data import stable_split

splits = stable_split(reviewed_records)
train_dataset = Dataset.from_list(splits['train']).remove_columns(['metadata'])
validation_dataset = Dataset.from_list(splits['validation']).remove_columns(['metadata'])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
# torch reports bf16 as supported on T4 (emulated); real support needs compute capability >= 8
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization, dtype=DTYPE, device_map={'': 0})
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM')
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    max_length=3072,  # evidence-JSON prompts are ~1.5-2k tokens; 1024 would cut off the answer
    gradient_checkpointing=True,
    bf16=USE_BF16,
    fp16=not USE_BF16,  # T4 has no bf16
    report_to='none',
    seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
# fp16 GradScaler cannot unscale bf16 grads, so keep every trainable (LoRA) weight in fp32
print('trainable dtypes before:', {str(p.dtype) for p in trainer.model.parameters() if p.requires_grad}, '| model dtype:', model.dtype)
for param in trainer.model.parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
trainer.train()
trainer.save_model(str(OUTPUT_DIR / 'adapter'))
print('Saved adapter to', OUTPUT_DIR / 'adapter')

## Use and evaluate
`generate(question, evidence=...)` gives an English summary; `tool_mode=True` returns a JSON tool request. The evaluation compares the base model with the adapter on the held-out split and writes `artifacts/llm/eval_report.json`.

In [ ]:
from src.llm.infer import generate
print(generate("Assess technical-reversal evidence for ASELS as of 2025-09-30.", evidence="E1: Data quality gate returned PASS_WITH_WARNINGS."))
print(generate("Check data quality for ASELS as of 2025-09-30.", tool_mode=True))

In [ ]:
%cd /content/bist-harness
!git pull
!python -m src.llm.infer --adapter $ADAPTER

In [ ]:
!zip -qr results.zip $ADAPTER artifacts/llm/eval_report.json
from google.colab import files; files.download("results.zip")